<div align="center" style="font-size:28px; font-weight:bold;">
MFE 409: Financial Risk Management
<br>Problem Set 8
<br>Valentin Haddad
<br>Due 3/8 before midnight
</div>

<br>

Cohort 2, Group 6:
Lee James, Moazzami Ali, Yu Aiden, Cai Jenny, Li Zehao, Guo Lucy

## 1 LTCM

### Please read in details the paper “Risk Management Lessons from Long-Term Capital Management” by Philippe Jorion (on BruinLearn Module 10). Write short essays answering the following questions.

#### 1. What was the broad trading strategy of LTCM?

#### Answer:
LTCM's broad strategy was relative-value or convergence arbitrage. It looked for pairs or groups of securities that were very similar in fundamental risk but temporarily priced apart, then bought the “cheap” instrument and shorted the “rich” one, betting that the spread would narrow over time. The paper's figures and discussion point to this pattern across major fixed-income spread relationships such as BAA-Treasury, mortgage-backed security versus Treasury, and Italian-German bond spreads, which are presented as core market relationships around LTCM's operating period.

In practice, that meant LTCM was not mainly making big directional bets on whether the whole stock or bond market would rise. Instead, it tried to earn small pricing discrepancies repeatedly across many markets, especially government bonds, credit, and related derivatives, and then used very high leverage to turn those small expected profits into large returns. The strategy depended on historical relationships reverting toward normal, diversified positions offsetting one another, and market liquidity remaining available long enough for convergence to occur.

#### 2. Why did they need so much leverage?

#### Answer:
They needed so much leverage because LTCM's trades were designed to capture very small pricing gaps between closely related securities, so the raw return on each unlevered trade was modest. Convergence arbitrage works by betting that spreads will narrow by a few basis points rather than by making large directional calls, so to turn many small mispricings into hedge-fund-level returns, LTCM scaled the positions up massively with borrowed money and derivatives.

The problem, as Jorion emphasizes through the fund's leverage and asset-growth discussion, is that leverage also magnified losses when spreads moved the wrong way instead of converging. Because these positions were marked to market daily, even temporary widening in spreads could create large capital losses and margin pressure long before the trades had time to “work,” which is why leverage was central both to LTCM's success and to its collapse.

#### 3. How did their demise happen?

#### Answer:
LTCM's demise happened when the market relationships it expected to converge instead moved violently the other way during the 1998 crisis, especially after Russia's default triggered a global flight to liquidity and quality. Spreads that LTCM expected to narrow instead widened sharply, so positions that looked hedged in normal times produced large simultaneous losses across many markets.

Because the fund was extremely leveraged, those adverse spread moves quickly eroded its equity base and created severe funding pressure. As losses mounted, counterparties became less willing to extend credit on the same terms, liquidity dried up, and LTCM was pushed toward forced liquidation at exactly the worst moment, which turned a bad mark-to-market shock into a near-fatal collapse.


#### 4. What were the most important issues with their risk management approach?

#### Answer:
The biggest problems in LTCM's risk management were that it treated historical relationships as more stable than they really were, underestimated the chance of extreme joint market moves, and relied too heavily on diversification that disappeared in stress. Jorion's discussion and figures on changing spread correlations and volatility show that risks that looked separate in normal periods became highly correlated during the crisis, so many positions lost money at the same time.

A second major weakness was insufficient attention to liquidity and funding risk relative to market risk. LTCM's models could tolerate ordinary price fluctuations, but they were far less prepared for a situation in which spreads widened sharply, market depth vanished, counterparties tightened financing, and the fund could not hold positions long enough for convergence, which meant leverage turned model error into a solvency threat.

#### 5. How would you manage risk for a fund trying to trade similar strategies?

#### Answer:
I would manage risk for a fund using LTCM-like strategies by assuming that convergence can take much longer than expected and that market relationships can break down exactly when positions are largest. That means using materially lower leverage, setting tighter concentration limits by trade theme rather than by instrument label, and requiring stress tests based on spread widening, correlation spikes, volatility jumps, and funding withdrawal rather than relying mainly on normal-period historical estimates.

I would also make liquidity and financing central parts of the risk framework, not side constraints. In practice, the fund should size positions so they can survive margin calls, maintain large liquidity buffers, diversify counterparties and funding sources, use drawdown and stop-loss triggers that force de-risking before capital is impaired, and regularly test whether the portfolio could be unwound under crisis conditions without assuming orderly markets.

## 2 Merton model for credit risk

### A company's equity is \$3 billion and the volatility of equity is 50%. The face value of debt is \$20 billion and time to debt maturity is 3 year. The risk-free rate is 4.5%. Make sure to show and explain all steps.

#### 1. What is the distance to default?

#### Answer:

We will use the Merton model here which treats a company's equity as a call option on its assets. The value of equity ($E$) and its volatility ($\sigma_E$) are functions of the firm's asset value ($V$), asset volatility ($\sigma_V$), the face value of debt ($F$), time to maturity ($T$), and the risk-free rate ($r$).

We can use these two equations to solve for distance to default:

1.  Equity Value: $E = V \cdot N(d_1) - F \cdot e^{-rT} \cdot N(d_2)$
2.  Equity Volatility: $\sigma_E \cdot E = N(d_1) \cdot V \cdot \sigma_V$

Where:

*   $N(x)$ is the cumulative standard normal distribution function.
*   $d_1 = \frac{\ln(V/F) + (r + \frac{1}{2}\sigma_V^2)T}{\sigma_V \sqrt{T}}$
*   $d_2 = d_1 - \sigma_V \sqrt{T}$

Since we are given $E$, $\sigma_E$, $F$, $T$, and $r$, we just need to solve for $V$ and $\sigma_V$ simultaneously using these two equations. We can do this in python using `scipy.optimize.fsolve` to find the values of $V$ and $\sigma_V$ that satisfy these equations. Then once these two values are found, we can solve for distance to default expression $\frac{\ln(V/F) + (r - \frac{1}{2}\sigma_V^2)T}{\sigma_V \sqrt{T}}$. Doing so gives $DD = 0.8939$

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import fsolve

E = 3e9
sigma_E = 0.50
F = 20e9
T = 3
r = 0.045

def merton_equations(params):
    V, sigma_V = params

    if sigma_V <= 0:
        return [1e10, 1e10]

    d1 = (np.log(V / F) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
    d2 = d1 - sigma_V * np.sqrt(T)

    # Equity Value
    eq1 = V * norm.cdf(d1) - F * np.exp(-r * T) * norm.cdf(d2) - E
    # Equity Volatility
    eq2 = (norm.cdf(d1) * V * sigma_V) / (sigma_E * E) - 1

    return [eq1, eq2]

initial_guess = [E + F, sigma_E * 0.5]

V_solution, sigma_V_solution = fsolve(merton_equations, initial_guess)

print(f"Calculated Asset Value (V): ${V_solution/1e9:.2f} billion")
print(f"Calculated Asset Volatility (sigma_V): {sigma_V_solution:.2%}")

V = V_solution
sigma_V = sigma_V_solution

d1 = (np.log(V / F) + (r + 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))
d2 = d1 - sigma_V * np.sqrt(T)

distance_to_default = (np.log(V / F) + (r - 0.5 * sigma_V**2) * T) / (sigma_V * np.sqrt(T))

print(f"d1: {d1:.4f}")
print(f"d2: {d2:.4f}")
print(f"Distance to Default: {distance_to_default:.4f}")

Calculated Asset Value (V): $20.22 billion
Calculated Asset Volatility (sigma_V): 8.71%
d1: 1.0447
d2: 0.8939
Distance to Default: 0.8939


### 2. What is the default probability?

#### Answer:

The default probability (PD) in the Merton model is given by $N(-d_2)$, where $N$ is the cumulative standard normal distribution function. Intuitively, this represents the probability that the asset value at maturity will fall below the face value of the debt. Doing so gives $PD = 0.1857$

In [ ]:
default_probability = norm.cdf(-d2)

print(f"Default Probability: {default_probability:.4f}")

Default Probability: 0.1857


### 3. What is the expected recovery rate on the debt?

#### Answer:

In the Merton model, the value of the firm's debt ($B$) can be found by subtracting the equity value ($E$) from the total asset value ($V$).

$B = V - E$

The expected recovery rate on the debt can then be approximated as the market value of the debt ($B$) divided by its face value ($F$). This represents the proportion of the face value of the debt that is expected to be recovered by debtholders. Doing so gives $ERR = 0.8612$

In [ ]:
market_value_of_debt = V - E

expected_recovery_rate = market_value_of_debt / F

print(f"Market Value of Debt (B): ${market_value_of_debt/1e9:.2f} billion")
print(f"Expected Recovery Rate: {expected_recovery_rate:.4f}")

Market Value of Debt (B): $17.22 billion
Expected Recovery Rate: 0.8612
